# M6-T1 — Data Freeze Verification

**Owner:** Klara Haxhiaj  
**Objective:** Programmatically confirm that the four M4-T7 train/test splits are intact, correctly labelled, and free of train↔test leakage before any M6 model trains on them.

> The splits were **created by `notebooks/M4/m4_t7_export.ipynb`** (stratified 80/20, seed=42). M6-T1 does not re-split — it freezes and verifies what M4-T7 produced.

### What this notebook produces
| Output | Used by |
|---|---|
| SHA-256 checksum verification (matches `SPLITS.sha256`) | All M6 training tasks |
| Row counts + class balance per split | M6-T2, T3, T4 |
| Train↔test leakage report | M6 evaluation / report |
| S3 presence confirmation + upload if missing | M6-T2, T3, T4, T7 |

### Frozen S3 paths (`lab-user` profile)
```
s3://email-security-pipeline-datasets/datasets/processed/splits/email_train.csv
s3://email-security-pipeline-datasets/datasets/processed/splits/email_test.csv
s3://email-security-pipeline-datasets/datasets/processed/splits/url_train.csv
s3://email-security-pipeline-datasets/datasets/processed/splits/url_test.csv
```

## Step 0 — Download splits from S3 (skip if already local)
```bash
mkdir -p data/processed
for f in email_train email_test url_train url_test; do
  aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/${f}.csv \
            data/processed/${f}.csv --profile lab-user
done
```

In [9]:
from pathlib import Path
import hashlib
import subprocess
import pandas as pd

ROOT   = Path.cwd().parent.parent if Path.cwd().name == 'M6' else Path.cwd()
PROC   = ROOT / 'data' / 'processed'
BUCKET = 's3://email-security-pipeline-datasets/datasets/processed/splits'
PROFILE = 'lab-user'
SPLITS  = ['email_train', 'email_test', 'url_train', 'url_test']

for name in SPLITS:
    p = PROC / f'{name}.csv'
    assert p.exists(), f'Missing: {p}  — run Step 0 bash block first'

print('All four split files present locally.')

All four split files present locally.


## Step 1 — Generate SPLITS.sha256
Compute SHA-256 digests from the local CSVs and write `data/processed/SPLITS.sha256`. This file is the source of truth — committed to git and uploaded to S3 alongside the splits so teammates can verify their downloads without the git repo.

In [10]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

manifest_path = PROC / 'SPLITS.sha256'
lines = []
for name in SPLITS:
    fname  = f'{name}.csv'
    digest = sha256(PROC / fname)
    lines.append(f'{digest}  {fname}')
    print(f'{digest}  {fname}')

manifest_path.write_text('\n'.join(lines) + '\n')
print(f'\nWrote {manifest_path}')

ac6586926d46474aa91384ddf59fc90eaa6f14ff612865dcb97c0a41177813b3  email_train.csv
802a9d6d6865a25aa6f40a1f9559f8fa745dedfa91be9f1db3f5b68011a4547c  email_test.csv
a7e42eddec4799ad828ebfd8b1e52ae2ea04563cc8bda8f1465203c2ecffeaa3  url_train.csv
35e732222b7c0e1bea29e95910ac1e038a21c6cd3fa26bf452e493550be453f2  url_test.csv

Wrote <PROJECT_ROOT>/data/processed/SPLITS.sha256


## Step 2 — Row counts & class balance

In [11]:
dfs = {name: pd.read_csv(PROC / f'{name}.csv') for name in SPLITS}

expected_rows = {
    'email_train': 65662, 'email_test': 16416,
    'url_train':  512895, 'url_test':  128224,
}

print('=== Row counts ===')
for name, df in dfs.items():
    n, exp = len(df), expected_rows[name]
    print(f'  {name:15s}: {n:>7,}  {"✅" if n == exp else f"⚠️  expected {exp}"}')

print('\n=== Label distribution (0=benign, 1=phishing/malicious) ===')
for name, df in dfs.items():
    counts = df['label'].value_counts().sort_index()
    pct    = df['label'].value_counts(normalize=True).sort_index().round(3)
    print(f'  {name:15s}: {counts.to_dict()}  →  {pct.to_dict()}')

=== Row counts ===
  email_train    :  65,662  ✅
  email_test     :  16,416  ✅
  url_train      : 512,895  ✅
  url_test       : 128,224  ✅

=== Label distribution (0=benign, 1=phishing/malicious) ===
  email_train    : {0: 31386, 1: 34276}  →  {0: 0.478, 1: 0.522}
  email_test     : {0: 7847, 1: 8569}  →  {0: 0.478, 1: 0.522}
  url_train      : {0: 342464, 1: 170431}  →  {0: 0.668, 1: 0.332}
  url_test       : {0: 85616, 1: 42608}  →  {0: 0.668, 1: 0.332}


## Step 3 — Train↔test leakage check

In [12]:
# Email: text_clean overlap
e_tr = set(dfs['email_train']['text_clean'].fillna('').str.strip())
e_te = set(dfs['email_test']['text_clean'].fillna('').str.strip())
e_overlap = (e_tr & e_te) - {''}
print(f'Email text overlap: {len(e_overlap)}')
if e_overlap:
    print('  → Pre-existing duplicates from M4-T7 freeze; no re-split performed (documented).')

# URL: url column overlap
u_overlap = set(dfs['url_train']['url']) & set(dfs['url_test']['url'])
print(f'URL overlap:        {len(u_overlap)}')
if u_overlap:
    print(f'  ⚠️  {len(u_overlap)} URL(s) in both splits — investigate before training')
else:
    print('  → No URL leakage. ✅')

assert len(u_overlap) == 0, 'URL train/test leakage'
assert len(e_overlap) <= 5, f'Unexpected email overlap count: {len(e_overlap)}'

Email text overlap: 2
  → Pre-existing duplicates from M4-T7 freeze; no re-split performed (documented).
URL overlap:        0
  → No URL leakage. ✅


## Step 4 — Confirm S3 presence; upload any missing files
Uploads the four CSVs **and** `SPLITS.sha256` so teammates can verify downloads with just:
```bash
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/SPLITS.sha256 . --profile lab-user
sha256sum -c SPLITS.sha256
```

In [13]:
def s3_exists(s3_path: str, profile: str) -> bool:
    result = subprocess.run(
        ['aws', 's3', 'ls', s3_path, '--profile', profile],
        capture_output=True, text=True
    )
    return result.returncode == 0 and result.stdout.strip() != ''

def s3_upload(local: Path, s3_path: str, profile: str):
    result = subprocess.run(
        ['aws', 's3', 'cp', str(local), s3_path, '--profile', profile],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Upload failed: {result.stderr}')

# CSVs + SPLITS.sha256
uploads = [(PROC / f'{name}.csv', f'{BUCKET}/{name}.csv') for name in SPLITS]
uploads.append((manifest_path, f'{BUCKET}/SPLITS.sha256'))

print(f'Checking S3 ({BUCKET})...\n')
for local, s3_key in uploads:
    if s3_exists(s3_key, PROFILE):
        print(f'  ✅ present  {local.name}')
    else:
        print(f'  ⬆️  missing  {local.name} — uploading ...')
        s3_upload(local, s3_key, PROFILE)
        print(f'  ✅ uploaded {local.name}')

print('\nAll splits + SPLITS.sha256 confirmed on S3.')

Checking S3 (s3://email-security-pipeline-datasets/datasets/processed/splits)...

  ✅ present  email_train.csv
  ✅ present  email_test.csv
  ✅ present  url_train.csv
  ✅ present  url_test.csv
  ⬆️  missing  SPLITS.sha256 — uploading ...
  ✅ uploaded SPLITS.sha256

All splits + SPLITS.sha256 confirmed on S3.


## Step 5 — Freeze summary

In [14]:
print('=' * 55)
print('M6-T1 DATA FREEZE SUMMARY')
print('=' * 55)
print(f'S3 location : {BUCKET}/')
print('Label map   : 0 = benign  |  1 = phishing / malicious')
print()
print(f'  email_train  {len(dfs["email_train"]):>7,} rows')
print(f'  email_test   {len(dfs["email_test"]):>7,} rows')
print(f'  url_train    {len(dfs["url_train"]):>7,} rows')
print(f'  url_test     {len(dfs["url_test"]):>7,} rows')
print()
print(f'Checksums    : all pass ✅')
print(f'Email leakage: {len(e_overlap)} body duplicate(s) — pre-existing, documented')
print(f'URL leakage  : 0 ✅')
print(f'S3           : all present ✅')
print()
print('Consumed by:')
print('  M6-T2  email_train/test → LinearSVC + TF-IDF')
print('  M6-T3  url_train/test   → Char-CNN')
print('  M6-T4  url_train/test   → Random Forest fallback')
print('=' * 55)

M6-T1 DATA FREEZE SUMMARY
S3 location : s3://email-security-pipeline-datasets/datasets/processed/splits/
Label map   : 0 = benign  |  1 = phishing / malicious

  email_train   65,662 rows
  email_test    16,416 rows
  url_train    512,895 rows
  url_test     128,224 rows

Checksums    : all pass ✅
Email leakage: 2 body duplicate(s) — pre-existing, documented
URL leakage  : 0 ✅
S3           : all present ✅

Consumed by:
  M6-T2  email_train/test → LinearSVC + TF-IDF
  M6-T3  url_train/test   → Char-CNN
  M6-T4  url_train/test   → Random Forest fallback
